# Stage 2: Above-Ground Biomass (AGB) Model Training

**Author** : Muhammad Wahyu Ramadhan
**GitHub**  : github.com/mwahyur46
**LinkedIn**: linkedin.com/in/mwahyur

Trains a Random Forest regressor to predict mangrove above-ground biomass
(AGB) from the full predictor stack plus the Stage 1 canopy height
prediction (`CH_m`), using GEDI L4A AGBD as the training target.

**Workflow steps:**
1. Load AGB training samples exported from `01_preprocessing.ipynb`
2. Exploratory check: distribution of AGBD target
3. Load the Stage 1 CH model and predict `CH_m` at each AGB sample location
4. Train-test split (70/30, `random_state=42`)
5. Hyperparameter tuning via RandomizedSearchCV (5-fold CV)
6. Evaluate on the held-out test set (R2, RMSE, MAE, Bias)
7. Diagnostic plots: 1:1 scatter, feature importance, Pearson correlation matrix
8. Save the trained model for use in Stage 3 (inference)

**Note on CH_m:** the AGB sample table exported from GEE does not include
`CH_m`, since the Stage 1 model is trained locally in Python (not in GEE).
`CH_m` is generated here by applying the saved Stage 1 model to the base
predictors at each AGB sample location -- this keeps Stage 1 and Stage 2
consistent with the same trained model used later for wall-to-wall
inference in `04_carbon_inference.ipynb`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/GitHub/mangrove-canopy-height-agb-estimation-gee/src')

import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

import model_utils as mu


## 1. Load Training Data


In [ ]:
DATA_PATH  = '/content/drive/MyDrive/GitHub/mangrove-canopy-height-agb-estimation-gee/data/raw/agb_samples_full.csv'
TARGET_COL = 'agbd'

df = pd.read_csv(DATA_PATH)

print(f'Loaded {len(df)} samples from: {DATA_PATH}')
df.head()


In [ ]:
# Base predictor columns (same feature set as Stage 1, without CH_m yet)
BASE_FEATURE_COLS = [
    'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12',
    'NDVI', 'NDWI', 'MNDWI', 'NDMI', 'CMRI', 'MVI', 'NDRE', 'SAVI', 'EVI',
    'VV', 'VH', 'slope',
]

missing = [c for c in BASE_FEATURE_COLS + [TARGET_COL] if c not in df.columns]
if missing:
    print(f'[WARNING] Missing columns: {missing}')
else:
    print('All expected base columns present.')


## 2. AGBD Target Distribution

Inspect the distribution of the GEDI L4A AGBD target before training.
Identifies outliers, skewness, and data coverage that may affect model
performance.


In [ ]:
print('--- AGBD Distribution Stats ---')
print(df[TARGET_COL].describe())
print(f'Skewness: {df[TARGET_COL].skew():.3f}')


In [ ]:
print(df['agbd'].describe())
print(f'Skewness: {df["agbd"].skew():.3f}')

In [ ]:
# Filter: pertahankan nilai dalam rentang biologis realistis
Q99 = df['agbd'].quantile(0.99)
print(f'99th percentile: {Q99:.2f} Mg/ha')

df_filtered = df[df['agbd'] <= Q99].copy()
df_filtered['CH_m'] = ch_model.predict(df_filtered[BASE_FEATURE_COLS].values)
print(f'Samples before : {len(df)}')
print(f'Samples after  : {len(df_filtered)}')
print(f'Dropped        : {len(df) - len(df_filtered)}')
print(df_filtered['agbd'].describe())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(df[TARGET_COL], bins=50, color='#2e7d32', edgecolor='white', linewidth=0.5)
axes[0].set_xlabel('AGBD (Mg/ha)')
axes[0].set_ylabel('Count')
axes[0].set_title('AGBD Distribution (GEDI L4A)')
axes[0].axvline(df[TARGET_COL].mean(), color='red', linestyle='--', linewidth=1.2, label=f'Mean: {df[TARGET_COL].mean():.1f}')
axes[0].axvline(df[TARGET_COL].median(), color='orange', linestyle='--', linewidth=1.2, label=f'Median: {df[TARGET_COL].median():.1f}')
axes[0].legend(fontsize=9)

# Boxplot
axes[1].boxplot(df[TARGET_COL], vert=True, patch_artist=True,
                boxprops=dict(facecolor='#a5d6a7', color='#2e7d32'),
                medianprops=dict(color='red', linewidth=2))
axes[1].set_ylabel('AGBD (Mg/ha)')
axes[1].set_title('AGBD Boxplot')

fig.tight_layout()
fig.savefig('/content/drive/MyDrive/GitHub/mangrove-canopy-height-agb-estimation-gee/images/agb_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('-' * 30)


## 3. Generate CH_m Predictor

Load the Stage 1 model and predict canopy height at each AGB sample
location, using the same base predictors. This becomes the `CH_m` feature
for Stage 2, consistent with the GEE workflow where the AGB model uses the
Stage 1 CH map as an additional predictor band.


In [ ]:
CH_MODEL_PATH = '/content/drive/MyDrive/GitHub/mangrove-canopy-height-agb-estimation-gee/models/ch_rf_model.pkl'

ch_model = joblib.load(CH_MODEL_PATH)
print(f'Loaded Stage 1 CH model: {CH_MODEL_PATH}')

df['CH_m'] = ch_model.predict(df[BASE_FEATURE_COLS].values)

print('CH_m predictor generated.')
print('-' * 30)
df[['CH_m']].describe()


In [ ]:
# Full AGB feature set: base predictors + CH_m
FEATURE_COLS = BASE_FEATURE_COLS + ['CH_m']

print(f'Number of features: {len(FEATURE_COLS)}')
print(f'Target column     : {TARGET_COL}')


In [ ]:
import json
import geopandas as gpd
import pandas as pd
import rasterio
from shapely.geometry import shape
import matplotlib.patches as mpatches

# ============================================================
# 2b. SPATIAL DISTRIBUTION OF AGB TRAINING SAMPLES
# ============================================================

STACK_PATH = '/content/drive/MyDrive/GitHub/mangrove-canopy-height-agb-estimation-gee/data/raw/base_stack_west_kalimantan.tif'
IMAGES_DIR = '/content/drive/MyDrive/GitHub/mangrove-canopy-height-agb-estimation-gee/images'

geometry = df['.geo'].apply(lambda x: shape(json.loads(x)))
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs='EPSG:4326')

with rasterio.open(STACK_PATH) as src:
    b8a = src.read(8)
    b11 = src.read(9)
    b4  = src.read(3)
    extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]
    crs_raster = src.crs

def normalize(band):
    p2, p98 = np.percentile(band[band > 0], [2, 98])
    return np.clip((band - p2) / (p98 - p2), 0, 1)

rgb = np.dstack([normalize(b8a), normalize(b11), normalize(b4)])
gdf_proj = gdf.to_crs(crs_raster)

bounds = gdf_proj.total_bounds
cols = np.linspace(bounds[0], bounds[2], 4)
rows = np.linspace(bounds[1], bounds[3], 3)

np.random.seed(42)
zoom_centers = []
for x0, x1 in zip(cols[:-1], cols[1:]):
    for y0, y1 in zip(rows[:-1], rows[1:]):
        cell = gdf_proj.cx[x0:x1, y0:y1]
        if len(cell) > 0:
            zoom_centers.append(cell.sample(1))

zoom_points = gpd.GeoDataFrame(pd.concat(zoom_centers), crs=gdf_proj.crs)

ZOOM_BUFFER  = 0.05
colors_zoom  = ['cyan', 'magenta', 'yellow', 'lime', 'orange', 'white']

fig, axes = plt.subplots(3, 2, figsize=(16, 21))
axes = axes.flatten()

ax = axes[0]
ax.imshow(rgb, extent=extent, origin='upper', aspect='equal')
scatter = ax.scatter(
    gdf_proj.geometry.x, gdf_proj.geometry.y,
    c=gdf_proj['agbd'], cmap='Greens',
    s=4, alpha=0.7, linewidths=0,
)
for i, (_, row) in enumerate(zoom_points.iterrows()):
    cx, cy = row.geometry.x, row.geometry.y
    rect = mpatches.Rectangle(
        (cx - ZOOM_BUFFER, cy - ZOOM_BUFFER),
        ZOOM_BUFFER * 2, ZOOM_BUFFER * 2,
        linewidth=1.5, edgecolor=colors_zoom[i], facecolor='none',
        label=f'Zoom {i + 1}'
    )
    ax.add_patch(rect)
cbar = fig.colorbar(scatter, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label('AGBD (Mg/ha)')
ax.set_title('Full Scene\n(B8A = Red, B11 = Green, B4 = Blue)')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.legend(loc='lower right', fontsize=8)

for i, (_, row) in enumerate(zoom_points.iterrows()):
    ax = axes[i + 1]
    cx, cy = row.geometry.x, row.geometry.y
    xmin, xmax = cx - ZOOM_BUFFER, cx + ZOOM_BUFFER
    ymin, ymax = cy - ZOOM_BUFFER, cy + ZOOM_BUFFER

    ax.imshow(rgb, extent=extent, origin='upper', aspect='equal')
    scatter_zoom = ax.scatter(
        gdf_proj.geometry.x, gdf_proj.geometry.y,
        c=gdf_proj['agbd'], cmap='Greens',
        s=20, alpha=0.8, linewidths=0,
    )
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)

    for spine in ax.spines.values():
        spine.set_edgecolor(colors_zoom[i])
        spine.set_linewidth(2)

    cbar_z = fig.colorbar(scatter_zoom, ax=ax, fraction=0.03, pad=0.02)
    cbar_z.set_label('AGBD (Mg/ha)')
    ax.set_title(f'Zoom {i + 1} (center: {cx:.2f}, {cy:.2f})')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')

fig.suptitle(
    'GEDI L4A AGB Training Samples on False Color Composite (B8A, B11, B4)',
    fontsize=14, y=1.01
)
fig.tight_layout()
for ax in axes:
    ax.yaxis.set_tick_params(rotation=90)
fig.savefig(f'{IMAGES_DIR}/agb_sample_spatial_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('-' * 30)
print(f'Total samples plotted : {len(gdf)}')
print(f'Raster CRS            : {crs_raster}')
print(f'Zoom panels           : {len(zoom_points)}')

## 4. Train-Test Split (70/30)


In [ ]:
X = df_filtered[FEATURE_COLS].values
y = df_filtered[TARGET_COL].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=mu.RANDOM_STATE
)

print(f'Train shape: X={X_train.shape}, y={y_train.shape}')
print(f'Test shape : X={X_test.shape}, y={y_test.shape}')


## 5. Hyperparameter Tuning (RandomizedSearchCV)

Random Forest with a standard search space over `n_estimators`, `max_depth`,
`min_samples_split`, `min_samples_leaf`, `max_features`, and `bootstrap`.
5-fold cross-validation, scored on R2.


In [ ]:
import time

start_time = time.time()

search = mu.train_rf_random_search(
    X_train, y_train,
    param_dist=mu.RF_PARAM_DIST,
    n_iter=100,
    cv=5,
    n_jobs=-1,
    scoring='r2',
    random_state=mu.RANDOM_STATE,
    verbose=2,
)

agb_model = search.best_estimator_

end_time = time.time()
print(f'Execution time: {(end_time - start_time):.2f} seconds')


## 6. Test Set Evaluation


In [ ]:
y_pred = agb_model.predict(X_test)

agb_metrics = mu.evaluate_regression(y_test, y_pred, unit='Mg/ha', label='AGB (Stage 2)')


## 7. Diagnostic Plots


In [ ]:
agb_importance = mu.get_feature_importance(agb_model, FEATURE_COLS)
agb_importance


In [ ]:
fig = mu.plot_model_diagnostics(
    y_test, y_pred, agb_importance,
    unit='Mg/ha', label='AGB (Stage 2)', metrics=agb_metrics,
)
fig.savefig('/content/drive/MyDrive/GitHub/mangrove-canopy-height-agb-estimation-gee/images/agb_model_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))
mu.plot_correlation_matrix(
    df, columns=FEATURE_COLS + [TARGET_COL],
    title='Pearson Correlation Matrix: AGB Predictors and Target', ax=ax,
)
fig.savefig('/content/drive/MyDrive/GitHub/mangrove-canopy-height-agb-estimation-gee/images/agb_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()


## 8. Save Model


In [ ]:
MODEL_PATH = '/content/drive/MyDrive/GitHub/mangrove-canopy-height-agb-estimation-gee/models/agb_rf_model.pkl'

joblib.dump(agb_model, MODEL_PATH)

print(f'Model saved: {MODEL_PATH}')
print(f'Best params: {search.best_params_}')


## Summary

| Metric | Value |
|--------|------:|
| R2     | see output above |
| RMSE   | see output above |
| MAE    | see output above |
| Bias   | see output above |

Proceed to `04_carbon_inference.ipynb` for wall-to-wall CH + AGB inference
and Stage 3 carbon stock derivation.
